# Q-ViewBone · Colab pilot
Clone code từ Git, đọc dữ liệu riêng trên Drive, lưu kết quả bền vững và chạy tiếp sau khi runtime ngắt.

Mặc định `dummy` chỉ kiểm tra pipeline, không tải weights và không tạo kết quả VLM. Đổi sang `qwen` và chọn **Runtime → Change runtime type → GPU** để chạy model thật. Cặp ảnh chưa được duyệt chỉ dùng khi chủ động bật `ALLOW_UNVERIFIED`; đầu ra mang nhãn exploratory. Không đưa dataset hoặc token lên Git.

In [ ]:
#@title 1. Git, dữ liệu và cấu hình thí nghiệm
REPO_URL = '' #@param {type:'string'}
GIT_BRANCH = 'main' #@param {type:'string'}
EXISTING_CHECKOUT = '' #@param {type:'string'}
USE_DRIVE = True #@param {type:'boolean'}
DATA_DIR = '/content/drive/MyDrive/QViewBone/vqa-dataset' #@param {type:'string'}
WORK_DIR = '/content/drive/MyDrive/QViewBone' #@param {type:'string'}
CACHE_IMAGES_LOCALLY = True #@param {type:'boolean'}
BACKEND = 'dummy' #@param ['dummy', 'qwen']
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct' #@param {type:'string'}
DEVICE = 'auto' #@param ['auto', 'cuda', 'cpu']
DTYPE = 'auto' #@param ['auto', 'float16', 'bfloat16', 'float32']
QUANTIZATION = 'none' #@param ['none', '4bit']
MAX_PIXELS = 401408 #@param {type:'integer'}
MAX_NEW_TOKENS = 128 #@param {type:'integer'}
MAX_PAIRS = 30 #@param {type:'integer'}
TASK_LIMIT = 3 #@param {type:'integer'}
SPLIT = 'train' #@param ['train', 'val', 'test', 'all']
RUN_NAME = 'smoke-dummy' #@param {type:'string'}
ALLOW_UNVERIFIED = False #@param {type:'boolean'}


In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
checkout = Path(EXISTING_CHECKOUT).expanduser().resolve() if EXISTING_CHECKOUT else Path('/content/qviewbone-code')
if not checkout.exists():
    if not REPO_URL:
        raise ValueError('Điền REPO_URL, hoặc EXISTING_CHECKOUT nếu đã clone repository.')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, REPO_URL, str(checkout)], check=True)
PROJECT = checkout if (checkout / 'src/qviewbone').is_dir() else checkout / 'QViewBone'
if not (PROJECT / 'src/qviewbone').is_dir():
    raise ValueError('Không tìm thấy QViewBone trong checkout.')
os.environ['QVB_ROOT'] = str(PROJECT)
# Nếu checkout đã tồn tại, giữ nguyên code để resume, không tự git pull.
subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], check=True)
install_target = str(PROJECT) + ('[vlm]' if BACKEND == 'qwen' else '')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', install_target], check=True)


In [ ]:
raw = Path(DATA_DIR).expanduser().resolve()
work = Path(WORK_DIR).expanduser().resolve()
if not (raw / 'qviewbone_crossview_candidates.jsonl').is_file():
    raise FileNotFoundError('DATA_DIR phải chứa các tệp JSONL/CSV và thư mục image-cache.')
images = raw / 'image-cache'
if not images.is_dir():
    raise FileNotFoundError('Chưa có image-cache. Copy ảnh gốc vào DATA_DIR trước khi chạy.')
if CACHE_IMAGES_LOCALLY:
    local_images = Path('/content/qviewbone-images')
    shutil.copytree(images, local_images, dirs_exist_ok=True)
    images = local_images
overrides = {'QVB_DATA_DIR': raw, 'QVB_IMAGE_DIR': images, 'QVB_PREPARED_DIR': work / 'prepared',
             'QVB_PAIR_REVIEW': work / 'pair_review.csv', 'QVB_OUTPUT_DIR': work / 'runs',
             'QVB_MODEL_ID': MODEL_ID, 'QVB_DEVICE': DEVICE, 'QVB_DTYPE': DTYPE,
             'QVB_QUANTIZATION': QUANTIZATION, 'QVB_MAX_PIXELS': MAX_PIXELS, 'QVB_MAX_NEW_TOKENS': MAX_NEW_TOKENS}
os.environ.update({k: str(v) for k,v in overrides.items()})
os.environ.setdefault('HF_HOME', '/content/hf-cache')
def cli(*args):
    subprocess.run([sys.executable, '-m', 'qviewbone', *map(str,args)], check=True)
cli('settings')
if BACKEND == 'qwen':
    from qviewbone.settings import load_settings
    from qviewbone.inference import hardware_settings
    print(hardware_settings(load_settings().model))
# Nếu cần HF_TOKEN, lấy từ Colab Secrets trong phiên; không ghi token vào notebook.


## Xác minh trước khi chạy thật
Lần đầu `prepare` tạo `WORK_DIR/pair_review.csv`. Điền hai cột xác nhận quan hệ cặp, `status=verified` và `approved_categories` (phân tách bằng dấu `;`) sau khi kiểm tra từng câu hỏi. Không sửa ID cặp. Chạy lại cell chuẩn bị sau khi lưu bảng.

Backend dummy được phép dùng ứng viên để smoke test; nhánh này chỉ tạo output giả. Backend qwen mặc định không dùng cặp chưa duyệt. Khi đổi model/config/dataset/GPU, chọn RUN_NAME mới; khi resume, dùng đúng settings và tên run cũ.

In [ ]:
exploratory = ALLOW_UNVERIFIED or BACKEND == 'dummy'
args = ['prepare', '--max-pairs', str(MAX_PAIRS)]
if exploratory:
    args.append('--allow-unverified')
    print('Chế độ smoke-test / thăm dò, không phải benchmark đã xác minh.')
cli(*args)
report = json.loads((work / 'prepared/preparation_report.json').read_text())
if report['tasks'] == 0:
    print('Chưa có task hợp lệ. Hoàn thành pair_review.csv rồi chạy lại cell này; model chưa được tải.')


In [ ]:
if report['tasks']:
    args = ['run', '--backend', BACKEND, '--run-name', RUN_NAME, '--split', SPLIT, '--limit', str(TASK_LIMIT)]
    if exploratory:
        args.append('--allow-unverified')
    cli(*args)
else:
    print('Bỏ qua inference vì chưa có task hợp lệ.')


In [ ]:
run_dir = work / 'runs' / RUN_NAME
if report['tasks'] and (run_dir / 'predictions.jsonl').is_file():
    cli('analyze', run_dir)
    print('Kết quả đã lưu:', run_dir)
    print((run_dir / 'metrics.csv').read_text(encoding='utf-8-sig'))
else:
    print('Chưa có predictions để phân tích.')
